In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
!pip install seaborn
import seaborn as sns

In [5]:
CLEAN_DATA_PATH = '../data/processed/train_clean.parquet'
EMBEDDINGS_PATH = '../data/raw/item_embeddings.npz'

df = pd.read_parquet(CLEAN_DATA_PATH)

print(f"Загружено строк: {len(df):,}")

Загружено строк: 40,618,962


Генерация "счетчиков"
популярность автора
активность пользователя

In [6]:
def create_agg_features(df):

    #Популярность автора - средний Watch Ratio
    author_stats = df.groupby('author_id')['target'].agg(['mean', 'count']).reset_index()
    author_stats.columns = ['author_id', 'author_mean_target', 'author_video_count']

    #Активность пользователя - сколько видео он посмотрел и его средний досмотр
    user_stats = df.groupby('user_id')['target'].agg(['mean', 'count']).reset_index()
    user_stats.columns = ['user_id', 'user_mean_target', 'user_watch_count']

    df = df.merge(author_stats, on='author_id', how='left')
    df = df.merge(user_stats, on='user_id', how='left')

    return df

df = create_agg_features(df)
print("Готово. Новые столбцы:", ['author_mean_target', 'user_mean_target'])

Готово. Новые столбцы: ['author_mean_target', 'user_mean_target']


Заменим Bool значения на 1,0 и создадим новую фичу как engagement_score:
$$EngagementScore = (Like + Share + Bookmark + ClickAuthor + OpenComments) - Dislike$$

In [7]:
def transform_binary_and_categories(df):

    #Список булевых столбцов
    bool_cols = ['like', 'dislike', 'share', 'bookmark', 'click_on_author', 'open_comments']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].astype(int)

    #Создаем комплексную фичу Engagement Score
    df['engagement_score'] = df[['like', 'share', 'bookmark', 'click_on_author', 'open_comments']].sum(axis=1)
    if 'dislike' in df.columns:
        df['engagement_score'] = df['engagement_score'] - df['dislike']

    print("Категориальные и бинарные признаки обработаны.")
    return df

df = transform_binary_and_categories(df)
display(df[['user_id', 'item_id', 'engagement_score', 'target']].head())

Категориальные и бинарные признаки обработаны.


,user_id,item_id,engagement_score,target
0,4498832,289321432,0,1.000000
1,3935673,524285128,0,0.136986
2,3591171,600360983,0,0.368421
3,1291951,134985354,0,1.000000
4,1069595,370435754,0,0.071429


Теперь у модели есть не только target, но и Engagement Score.

Если юзер посмотрел видео на 20% (target=0.2), но при этом поставил лайк и сделал репост (engagement_score=2), модель поймет, что это видео было ОЧЕНЬ интересным, просто коротким или прерванным звонком.

Займемся агрегированными признаками и посчитаем средние показатели для каждого author_id и user_id.

In [8]:
def create_stats_features(df):

    author_stats = df.groupby('author_id').agg({
        'target': 'mean',
        'engagement_score': 'mean',
        'item_id': 'count'
    }).reset_index()

    author_stats.columns = ['author_id', 'author_avg_target', 'author_avg_engagement', 'author_popularity']

    user_stats = df.groupby('user_id').agg({
        'target': 'mean',
        'engagement_score': 'mean',
        'item_id': 'count'
    }).reset_index()

    user_stats.columns = ['user_id', 'user_avg_target', 'user_avg_engagement', 'user_activity']

    df = df.merge(author_stats, on='author_id', how='left')
    df = df.merge(user_stats, on='user_id', how='left')

    print("Статистики успешно добавлены.")
    return df

df = create_stats_features(df)
display(df[['author_id', 'author_avg_target', 'user_id', 'user_avg_target']].head())

Статистики успешно добавлены.


,author_id,author_avg_target,user_id,user_avg_target
0,285593,0.648670,4498832,0.680623
1,239962,0.527395,3935673,0.482548
2,586575,0.572725,3591171,0.651666
3,1051229,0.548375,1291951,0.590401
4,702084,0.481873,1069595,0.556976


author_avg_engagement: Если у автора этот показатель высокий, значит, его контент провоцирует людей на действия. Модель быстро поймет, что его видео хайповые.

user_avg_target: Есть пользователи, которые смотрят всё до конца, а есть листатели, которые закрывают всё через секунду. Без этой фичи модель будет штрафовать видео за то, что листатель его закрыл, а теперь она поймет: «А, ну этот юзер всегда так делает, видео тут ни при чем».

Также я создал категорию «Длительность видео», так как люди по-разному смотрят короткие (5 сек) и длинные (60 сек) ролики.

In [9]:
def create_video_type_features(df):
    # 0-15 сек: Shorts, 15-30: Medium, 30+: Long
    df['video_duration_type'] = pd.cut(df['duration'],
                                       bins=[0, 15, 30, 300],
                                       labels=[1, 2, 3]).astype(int)

    print("Признак типа длительности видео создан.")
    return df

df = create_video_type_features(df)

Признак типа длительности видео создан.


Проверяем данные еще раз отдельно на Nun и сохраняем в новую БД

In [10]:
from src.data_loader import save_processed_data

if df.isnull().values.any():
    print("Заполнение пропусков в новых признаках...")
    df = df.fillna(0)

OUTPUT_DIR = '../data/processed/'
FILE_NAME = 'train_features.parquet'

save_processed_data(df, OUTPUT_DIR, FILE_NAME)
print(f"Итоговая таблица: {df.shape[0]:,} строк на {df.shape[1]} колонок")

Файл сохранен: ../data/processed/train_features.parquet
Итоговая таблица: 40,618,962 строк на 32 колонок
